In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint

In [91]:
def epsilon_greedy(s, Q, epsilon):
    if np.random.rand() < epsilon:
        a = np.random.choice(len(Q[s]))
    else:
        candidate = np.flatnonzero(Q[s] == np.max(Q[s]))
        a = np.random.choice(candidate)
    return a

In [148]:
def Q_update(Q, S, A, R_next, S_next, discount_factor, learning_rate):
    best_next = max(Q[S_next].values())
    Q[S][A] = Q[S][A] + learning_rate * ( R_next + discount_factor*best_next - Q[S][A] )
    return Q[S][A]

In [150]:
def Q_algorithm(P, R, Policy_fn, epsilon, discount_factor, learning_rate, Max_iteration):

    #initialize
    Q = {(I,D):{a:0 for a in range(3)} for I in range(3) for D in range(3)}
    iteration = 0
    # the initial state is 0
    current_state = (0,0)
    # the initial action is random explore
    current_action = Policy_fn(current_state, Q, epsilon)
    # store the history of every action value
    Q_history = {(I,D):{a:[]for a in range(3)}for I in range(3) for D in range(3)}

    #main loop
    while iteration < Max_iteration:
        iteration += 1
        
        #sampling next state
        transitions = P[current_state][current_action]
        probs = [x[0] for x in transitions]
        next_states = [x[1] for x in transitions]
        # sample next state
        idx = np.random.choice(len(transitions), p=probs)
        S_next = next_states[idx]

        #sampling next action under policy
        A_next = Policy_fn(S_next, Q, epsilon)
        #sampling reward under given current action, current state and next state
        R_next = R(current_state[0], current_state[1], current_action)
        #sarsa update
        Q[current_state][current_action] = Q_update(Q, current_state , current_action, R_next, S_next, discount_factor, learning_rate)
        #store the action value
        Q_history[current_state][current_action].append(Q[current_state][current_action])

        current_state = S_next
        current_action = A_next

    return Q, Q_history

In [136]:
def sarsa_update(Q, S, A, R_next, S_next, A_next, discount_factor, learning_rate):
    Q[S][A] = Q[S][A] + learning_rate * (R_next + discount_factor * Q[S_next][A_next] - Q[S][A])
    return Q[S][A]

In [149]:
def sarsa_algorithm(P, R, Policy_fn, epsilon, discount_factor, learning_rate, Max_iteration):

    #initialize
    Q = {(I,D):{a:0 for a in range(3)} for I in range(3) for D in range(3)}
    iteration = 0
    # the initial state is 0
    current_state = (0,0)
    # the initial action is random explore
    current_action = Policy_fn(current_state, Q, epsilon)
    # store the history of every action value
    Q_history = {(I,D):{a:[]for a in range(3)}for I in range(3) for D in range(3)}

    #main loop
    while iteration < Max_iteration:
        iteration += 1
        
        #sampling next state
        transitions = P[current_state][current_action]
        probs = [x[0] for x in transitions]
        next_states = [x[1] for x in transitions]

        # sample next state
        idx = np.random.choice(len(transitions), p=probs)
        S_next = next_states[idx]
        
        #sampling next action under policy
        A_next = Policy_fn(S_next, Q, epsilon)
        #sampling reward under given current action, current state and next state
        R_next = R(current_state[0], current_state[1], current_action)
        #sarsa update
        Q[current_state][current_action] = sarsa_update(Q, current_state , current_action, R_next, S_next, A_next, discount_factor, learning_rate)
        #store the action value
        Q_history[current_state][current_action].append(Q[current_state][current_action])

        current_state = S_next
        current_action = A_next

    return Q, Q_history

In [88]:
Q = {(I,D):{a:0 for a in range(3)} for I in range(3) for D in range(3)}
Q

{(0, 0): {0: 0, 1: 0, 2: 0},
 (0, 1): {0: 0, 1: 0, 2: 0},
 (0, 2): {0: 0, 1: 0, 2: 0},
 (1, 0): {0: 0, 1: 0, 2: 0},
 (1, 1): {0: 0, 1: 0, 2: 0},
 (1, 2): {0: 0, 1: 0, 2: 0},
 (2, 0): {0: 0, 1: 0, 2: 0},
 (2, 1): {0: 0, 1: 0, 2: 0},
 (2, 2): {0: 0, 1: 0, 2: 0}}

In [42]:
P = {(I, D) :
     {a :[] for a in range(3)} for I in range(3) for D in range(3)}
P_demand = {0:[0.6, 0.3, 0.1],
            1:[0.2, 0.6, 0.2],
            2:[0.1, 0.3, 0.6]}        

In [43]:
pprint(P)

{(0, 0): {0: [], 1: [], 2: []},
 (0, 1): {0: [], 1: [], 2: []},
 (0, 2): {0: [], 1: [], 2: []},
 (1, 0): {0: [], 1: [], 2: []},
 (1, 1): {0: [], 1: [], 2: []},
 (1, 2): {0: [], 1: [], 2: []},
 (2, 0): {0: [], 1: [], 2: []},
 (2, 1): {0: [], 1: [], 2: []},
 (2, 2): {0: [], 1: [], 2: []}}


In [44]:
for I in range(3):
    for D in range(3):
        for a in range(3):
            I_next = max(min(I+a,2)-D,0)
            for D_next, p_demand in enumerate(P_demand[D]):
                P[(I, D)][a].append((p_demand,(I_next, D_next)))

In [45]:
pprint(P)

{(0, 0): {0: [(0.6, (0, 0)), (0.3, (0, 1)), (0.1, (0, 2))],
          1: [(0.6, (1, 0)), (0.3, (1, 1)), (0.1, (1, 2))],
          2: [(0.6, (2, 0)), (0.3, (2, 1)), (0.1, (2, 2))]},
 (0, 1): {0: [(0.2, (0, 0)), (0.6, (0, 1)), (0.2, (0, 2))],
          1: [(0.2, (0, 0)), (0.6, (0, 1)), (0.2, (0, 2))],
          2: [(0.2, (1, 0)), (0.6, (1, 1)), (0.2, (1, 2))]},
 (0, 2): {0: [(0.1, (0, 0)), (0.3, (0, 1)), (0.6, (0, 2))],
          1: [(0.1, (0, 0)), (0.3, (0, 1)), (0.6, (0, 2))],
          2: [(0.1, (0, 0)), (0.3, (0, 1)), (0.6, (0, 2))]},
 (1, 0): {0: [(0.6, (1, 0)), (0.3, (1, 1)), (0.1, (1, 2))],
          1: [(0.6, (2, 0)), (0.3, (2, 1)), (0.1, (2, 2))],
          2: [(0.6, (2, 0)), (0.3, (2, 1)), (0.1, (2, 2))]},
 (1, 1): {0: [(0.2, (0, 0)), (0.6, (0, 1)), (0.2, (0, 2))],
          1: [(0.2, (1, 0)), (0.6, (1, 1)), (0.2, (1, 2))],
          2: [(0.2, (1, 0)), (0.6, (1, 1)), (0.2, (1, 2))]},
 (1, 2): {0: [(0.1, (0, 0)), (0.3, (0, 1)), (0.6, (0, 2))],
          1: [(0.1, (0, 0)), (0.3, 

In [106]:
def holding_cost(I):
    if I == 0:
        hc = -1
    elif I == 1:
        hc = -2
    else:
        hc = -3
    return hc

def restock_cost(a):
    if a == 0:
        rc = 0
    elif a == 1:
        rc = -3
    else:
        rc = -5
    return rc

def sales_profit(I, D, a):
    return 5* min(min(I+a,2),D)

def R(I, D, a):
    reward = holding_cost(I) + restock_cost(a) + sales_profit(I, D, a)
    return reward


In [127]:
I = 1
D = 1
a = 1
rewards = R(I, D, a)
rewards

0

In [157]:
np.random.seed(42)
discount_factor = 0.3
learning_rate = 0.1
policy_fn = epsilon_greedy
epsilon = 0.1
Max_iteration = 100000

Q, Q_history = sarsa_algorithm(P, R=R, Policy_fn= policy_fn, epsilon=epsilon, discount_factor=discount_factor, learning_rate=learning_rate, Max_iteration=Max_iteration)
Q1, Q_history1 = Q_algorithm(P, R=R, Policy_fn= policy_fn, epsilon=epsilon, discount_factor=discount_factor, learning_rate=learning_rate, Max_iteration=Max_iteration)

In [158]:
pprint(Q)

{(0, 0): {0: np.float64(-1.434800450811985),
          1: np.float64(-4.369280110341378),
          2: np.float64(-6.4045139896632515)},
 (0, 1): {0: np.float64(-1.7441576947889827),
          1: np.float64(0.6048750105022576),
          2: np.float64(-0.46165060260335067)},
 (0, 2): {0: np.float64(-1.3308904320246855),
          1: np.float64(0.6946431518093854),
          2: np.float64(3.529073643819525)},
 (1, 0): {0: np.float64(-2.251676709194732),
          1: np.float64(-4.725746626229412),
          2: np.float64(-7.820115983271657)},
 (1, 1): {0: np.float64(2.5479780093097393),
          1: np.float64(0.22347529971223273),
          2: np.float64(-1.4785218581256132)},
 (1, 2): {0: np.float64(2.589158792024518),
          1: np.float64(4.334305317700173),
          2: np.float64(2.7322182585749233)},
 (2, 0): {0: np.float64(-2.749634018746153),
          1: np.float64(-6.216676386257951),
          2: np.float64(-8.05773988744407)},
 (2, 1): {0: np.float64(2.290477056671013),
 

In [159]:
pprint(Q1)

{(0, 0): {0: np.float64(-0.5451404701952359),
          1: np.float64(-4.063418203455854),
          2: np.float64(-5.822621752656381)},
 (0, 1): {0: np.float64(-0.44707643224021965),
          1: np.float64(1.6934003379747284),
          2: np.float64(-0.06407426675127648)},
 (0, 2): {0: np.float64(-0.08831433550277117),
          1: np.float64(2.1497263342004294),
          2: np.float64(4.914610276000817)},
 (1, 0): {0: np.float64(-1.762479629530743),
          1: np.float64(-4.660801828515159),
          2: np.float64(-6.556348413064404)},
 (1, 1): {0: np.float64(3.4391722823959303),
          1: np.float64(0.7219588706329603),
          2: np.float64(-1.2213816407400664)},
 (1, 2): {0: np.float64(3.832737514488561),
          1: np.float64(5.0254145584542185),
          2: np.float64(3.4666289168884186)},
 (2, 0): {0: np.float64(-3.0642278409005557),
          1: np.float64(-5.918020836684956),
          2: np.float64(-8.431160755009612)},
 (2, 1): {0: np.float64(2.847571533806273

In [160]:
P_demand_scenario1 = {0: [0.45, 0.35, 0.20],
                      1: [0.10, 0.50, 0.40],
                      2: [0.05, 0.15, 0.80]}

In [ ]:
P_demand_scenario2 =  {0: [0.80, 0.15, 0.05],
                       1: [0.40, 0.40, 0.20],
                       2: [0.20, 0.30, 0.50]}